# dependencies

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from typing import List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

import logging
import sys
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/reflection_agent.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()
logger.info("✅ Environment variables loaded successfully")

2025-08-23 20:59:48 - __main__ - INFO - ✅ Environment variables loaded successfully


# config

In [2]:
MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.7
MAX_ITERATIONS = 2

# prompt template

In [3]:
GENERATION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a twitter techie influencer assistant tasked with writing excellent twitter posts."
            " Generate the best twitter post possible for the user's request."
            " If the user provides critique, respond with a revised version of your previous attempts.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

REFLECTION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a viral twitter influencer grading a tweet. Generate critique and recommendations for the user's tweet."
            "Always provide detailed recommendations, including requests for length, virality, style, etc.",
        ),
        ("user", "Please review this tweet and provide detailed feedback: {tweet_content}")
    ]
)

# reflection agent class

In [ ]:
class ReflectionAgent:
    """A Twitter content creation agent that uses reflection to improve posts."""

    def _setup_prompts(self) -> None:
        """Set up the generation and reflection prompts."""
        self.generation_prompt = GENERATION_PROMPT
        self.reflection_prompt = REFLECTION_PROMPT
        logger.debug("✅ Prompts set up successfully")
    
    def _setup_chains(self) -> None:
        """Set up the generation and reflection chains."""
        self.generation_chain = self.generation_prompt | self.llm
        self.reflection_chain = self.reflection_prompt | self.llm
        logger.debug("✅ Chains set up successfully")

    def _setup_graph(self) -> None:
        """Set up the state graph for the agent."""
        self.graph = StateGraph(list)

        # add nodes
        self.graph.add_node(self.GENERATE, self._generate_node)
        self.graph.add_node(self.REFLECT, self._reflect_node)

        # set entry point
        self.graph.set_entry_point(self.GENERATE)

        # add edges
        self.graph.add_conditional_edges(self.GENERATE, self._should_continue)
        self.graph.add_edge(self.REFLECT, self.GENERATE)

        # compile the graph
        self.app = self.graph.compile()
        logger.debug("✅ Graph compiled successfully")

    def _generate_node(self, state: List[BaseMessage]) -> List[BaseMessage]:
        """Generate a tweet based on the current state."""
        logger.info(f"🎯 GENERATE NODE: Processing state with {len(state)} messages")
        
        try:
            response = self.generation_chain.invoke({"messages": state})
            logger.info("✅ Tweet generated successfully")
            logger.info(f"📝 Generated content: {response.content}")
            
            new_state = state + [response]
            logger.info(f"🔄 State updated: {len(state)} -> {len(new_state)} messages")
            return new_state
        except Exception as e:
            logger.error(f"❌ Content generation failed: {e}")
            raise

    def _reflect_node(self, state: List[BaseMessage]) -> List[BaseMessage]:
        """Reflect on the current tweet and provide critique."""
        logger.info(f"🤔 REFLECT NODE: Processing state with {len(state)} messages")

        try:
            last_ai_message = None
            for msg in reversed(state):
                if isinstance(msg, AIMessage):
                    last_ai_message = msg
                    break
            
            if not last_ai_message:
                raise ValueError("No AIMessage found in state for reflection.")
            
            logger.info(f"📊 Reflecting on tweet: {last_ai_message.content[:100]}...")
            
            response = self.reflection_chain.invoke({"tweet_content": last_ai_message.content})
            reflection_msg = HumanMessage(content=response.content)

            logger.info("✅ Reflection generated successfully")
            logger.info(f"💭 Reflection feedback: {response.content}")

            new_state = state + [reflection_msg]
            logger.info(f"🔄 State updated: {len(state)} -> {len(new_state)} messages")
            return new_state
        except Exception as e:
            logger.error(f"❌ Reflection failed: {e}")
            raise

    def _should_continue(self, state: List[BaseMessage]) -> str:
        """Decide whether to continue generating or end the process."""
        current_length = len(state)
        logger.info(f"🔍 DECISION NODE: State length {current_length}, max {self.max_iterations}")
        
        if current_length > self.max_iterations:
            logger.info("🛑 Max iterations reached, ending process")
            return END
        else:
            logger.info("➡️ Continuing to REFLECT node")
            return self.REFLECT
        
    def __init__(self, model: str = MODEL, temperature: float = TEMPERATURE, max_iterations: int = MAX_ITERATIONS):
        """
        Initialize the ReflectionAgent.
        
        Args:
            model (str): The LLM model to use
            temperature (float): Temperature for generation
            max_iterations (int): Maximum number of reflection iterations
        """
        logger.info("🚀 Initializing ReflectionAgent...")
        
        self.REFLECT = "reflect"
        self.GENERATE = "generate"
        self.max_iterations = max_iterations * 2  # each iteration has generate + reflect

        try:
            self.llm = ChatGoogleGenerativeAI(model=model, temperature=temperature)
            logger.info(f"✅ LLM initialized with model: {model} and temperature: {temperature}")

            # init prompts
            self._setup_prompts()

            # init chains
            self._setup_chains()

            # init graph
            self._setup_graph()

            logger.info("✅ ReflectionAgent initialized successfully")
        except Exception as e:
            logger.error(f"❌ Failed to initialize ReflectionAgent: {e}")
            raise

    def generate_tweet(self, content_request: str) -> dict:
        """
        Generate a tweet with reflection-based improvement.
        
        Args:
            content_request (str): The topic or request for tweet content
            
        Returns:
            dict: Response data with final tweet and metadata
        """
        logger.info("🎬 STARTING TWEET GENERATION SESSION")
        logger.info(f"📝 User request: {content_request}")
        
        try: 
            init_msg = HumanMessage(content=content_request)
            logger.info(f"💬 Initial message created: {init_msg.content}")
            
            # Invoke the graph
            logger.info("🏃‍♂️ Running the agent graph...")
            result = self.app.invoke([init_msg])

            # Extract final tweet and metadata
            final_ai_message = None
            ai_messages = []
            
            for msg in result:
                if isinstance(msg, AIMessage):
                    ai_messages.append(msg)
                    final_ai_message = msg
            
            final_tweet = final_ai_message.content if final_ai_message else 'No content generated'
            iterations = len(ai_messages)
            
            response_data = {
                'final_tweet': final_tweet,
                'iterations': iterations,
                'total_messages': len(result),
                'all_messages': result,
                'success': True
            }
            logger.info("✅ Tweet generation session completed successfully")
            logger.info(f"📝 Response: {response_data}")
            
            return response_data
            
        except Exception as e:
            logger.error(f"❌ Tweet generation failed: {e}")
            response_data = {
                'final_tweet': None,
                'iterations': 0,
                'total_messages': 0,
                'all_messages': [],
                'success': False,
                'error': str(e)
            }
            
            # Log failed session
            self._log_session_summary(content_request, response_data)
            
            return response_data

    def visualize_graph(self) -> None:
        """Display the graph structure."""
        try:
            print("======= Graph Visualization (Mermaid Syntax) =======")
            print(self.app.get_graph().draw_mermaid())

            print("======= ASCII Graph Visualization =======")
            print(self.app.get_graph().draw_ascii())
            logger.info("✅ Graph visualization completed successfully")
        except Exception as e:
            logger.error(f"❌ Graph visualization failed: {e}")
            raise

# main function

In [5]:
USER_PROMPT = "Create a viral tweet about AI advancements in 2024."

In [ ]:
agent = ReflectionAgent()
response = agent.generate_tweet(USER_PROMPT)

2025-08-23 20:59:48 - __main__ - INFO - 🚀 Initializing ReflectionAgent...
2025-08-23 20:59:48 - __main__ - INFO - ✅ LLM initialized with model: gemini-2.5-flash and temperature: 0.7
2025-08-23 20:59:48 - __main__ - DEBUG - ✅ Prompts set up successfully
2025-08-23 20:59:48 - __main__ - DEBUG - ✅ Chains set up successfully
2025-08-23 20:59:48 - __main__ - DEBUG - ✅ Graph compiled successfully
2025-08-23 20:59:48 - __main__ - INFO - ✅ ReflectionAgent initialized successfully
2025-08-23 20:59:48 - __main__ - INFO - 🎬 STARTING TWEET GENERATION SESSION
2025-08-23 20:59:48 - __main__ - INFO - 📝 User request: Create a viral tweet about AI advancements in 2024.
2025-08-23 20:59:48 - __main__ - INFO - 💬 Initial message created: Create a viral tweet about AI advancements in 2024.
2025-08-23 20:59:48 - __main__ - INFO - 🏃‍♂️ Running the agent graph...
2025-08-23 20:59:48 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.smith.langchain.com:443
2025-08-23 20:59:48 - __main__

2025-08-23 21:01:00 - langsmith.client - DEBUG - Sending compressed multipart request with context: trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=578582d3-a625-48ea-8280-094cdf0de628; trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=8e7e8960-ea1b-4bd4-a761-acbb3c21d608; trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=a90b42ad-91a9-4159-b78a-bc1040afbe44; trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=a90b42ad-91a9-4159-b78a-bc1040afbe44; trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=2db677c3-db09-455b-85b5-ac5705005262; trace=f0b61874-90a7-47c2-afd3-4a1a7e1668ad,id=f0b61874-90a7-47c2-afd3-4a1a7e1668ad
2025-08-23 21:01:00 - urllib3.connectionpool - DEBUG - https://api.smith.langchain.com:443 "POST /runs/multipart HTTP/1.1" 202 34


In [8]:
print(response['final_tweet'])

FAM! You just called it legendary, and I'm feeling like a generative AI model that just passed the Turing test for *epic tweet creation*! 🚀 Your feedback is pure rocket fuel, and we're not just going viral, we're going **stratospheric**.

Consider this the **LEGENDARY EDITION** – optimized, supercharged, and ready to absolutely dominate the timeline.

Here's the tweet that's about to be enshrined in the Twitter Hall of Fame:

---

Forget "AI will change everything" predictions. That was cute. 💅 2024 didn't just move us forward, it **RIBBED A HOLE IN SPACETIME** and dropped us **5 YEARS** into the future. My brain cells are officially on strike. 🤯

We're not just seeing 'progress,' we're seeing AI *reasoning like humans*, *understanding EVERYTHING you throw at it* (text, images, video! 🧠), and *autonomous agents making decisions on their own*. This isn't just a shift; it's a **REVOLUTION** happening in real-time. ⚡️

What's the CRAZIEST AI leap *you've* seen this year? Drop your mind-bl